In [2]:
import pandas as pd

In [147]:
race_map = {
    'black': 'Black',
    'latino': 'Latino',
    'other': 'Other'
}

In [148]:
def cnt_effective(plans, enacted, minorities, state = 'Georgia'):
    minority_cnt = {}
    for minority in minorities:
        minority_cnt[minority] = 0
    for _, plan in plans.iterrows():
        for minority in minorities:
            cap_min = race_map[minority]
            
            enacted_num = enacted.loc[enacted['Demographic']==cap_min, 'Effective_Districts'].iloc[0]
            score = plan[f'{minority}_effective_score_cnt'] 
            
            if score >= enacted_num:
                minority_cnt[minority] += 1
    return minority_cnt 

In [176]:
def cnt_rough(plans, enacted_rough, proportions, minorities, total_district=14, state='Georgia'):
    minority_cnt = {}
    for minority in minorities:
        minority_cnt[minority] = 0
        
    for _, plan in plans.iterrows():
        for minority in minorities:
            num_effective = plan[f'{minority}_effective_score_cnt']
            cap_min = race_map[minority]
            proportion = proportions.loc['state_wide'][cap_min]
            rough = (num_effective/total_district)/proportion
            enacted = enacted_rough.loc[cap_min, state]
            if rough >= enacted:
                minority_cnt[minority] += 1
                
    return minority_cnt

In [150]:
def cnt_both(plans, enacted_effective, enacted_rough, proportions, minorities, total_district=14, state='Georgia'):
    minority_cnt = {}
    for minority in minorities:
        minority_cnt[minority] = 0
        
    for _, plan in plans.iterrows():
        for minority in minorities:
            cap_min = race_map[minority]
            
            enacted_num_eff = enacted_effective.loc[enacted_effective['Demographic']==cap_min, 'Effective_Districts'].iloc[0]
            num_effective = plan[f'{minority}_effective_score_cnt']
            
            if num_effective < enacted_num_eff:
                continue
            
            proportion = proportions.loc['state_wide'][cap_min]
            rough = (num_effective/total_district)/proportion
            enacted_r = enacted_rough.loc[cap_min, state]
            
            if rough >= enacted_r:
                minority_cnt[minority]+=1
            
    return minority_cnt
        

In [151]:
enacted_rough_porportion = pd.read_json('../../preprocessing/output/rough_proportionality.json')

# Georgia

In [177]:
ga_rb = pd.read_json('../outputs/Georgia/ga_raceblind_5000.jsonl', lines=True)
ga_vra = pd.read_json('../outputs/Georgia/ga_vra_5000.jsonl', lines=True)
ga_enacted = pd.read_json('../../preprocessing/output/Georgia/ga_enacted_effective.json')
ga_proportions = pd.read_json('../../preprocessing/output/Georgia/ga_cvap_proportion.json')

/var/folders/_7/0wq2xjx96vngcxdnd_qxp1z40000gn/T/ipykernel_30337/1470110957.py:4: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  ga_proportions = pd.read_json('../../preprocessing/output/Georgia/ga_cvap_proportion.json')
/var/folders/_7/0wq2xjx96vngcxdnd_qxp1z40000gn/T/ipykernel_30337/1470110957.py:4: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  ga_proportions = pd.read_json('../../preprocessing/output/Georgia/ga_cvap_proportion.json')
/var/folders/_7/0wq2xjx96vngcxdnd_qxp1z40000gn

In [178]:
ga_minority = ['black', 'latino','other']
ga_rb_effective_cnt = cnt_effective(ga_rb, ga_enacted, minorities=ga_minority, state='Georgia')
ga_rb_rough_cnt = cnt_rough(ga_rb, enacted_rough_porportion, ga_proportions, minorities=ga_minority)
ga_rb_both = cnt_both(ga_rb, ga_enacted, enacted_rough_porportion, ga_proportions, minorities=ga_minority)
print(f'enactedThreshold:{ga_rb_effective_cnt}, proportionalThreshold:{ga_rb_rough_cnt}, bothThreshold:{ga_rb_both}')

enactedThreshold:{'black': 1743, 'latino': 5000, 'other': 5000}, proportionalThreshold:{'black': 1743, 'latino': 5000, 'other': 5000}, bothThreshold:{'black': 1743, 'latino': 5000, 'other': 5000}


In [156]:
ga_vra_effective_cnt = cnt_effective(ga_vra, ga_enacted, minorities=ga_minority,state='Georgia')
ga_vra_rough_cnt = cnt_rough(ga_vra, enacted_rough_porportion, ga_proportions, minorities=ga_minority)
ga_vra_both = cnt_both(ga_vra, ga_enacted, enacted_rough_porportion, ga_proportions, minorities=ga_minority)
print(f'enactedThreshold:{ga_vra_effective_cnt}, proportionalThreshold:{ga_vra_rough_cnt}, bothThreshold:{ga_vra_both}')

enactedThreshold:{'black': 5000, 'latino': 5000, 'other': 5000}, proportionalThreshold:{'black': 5000, 'latino': 5000, 'other': 5000}, bothThreshold:{'black': 5000, 'latino': 5000, 'other': 5000}


In [158]:
import json

ga_result = {
    "groups": {
        "BLACK": {
            "enactedThreshold": {
                "raceBlind": ga_rb_effective_cnt['black'],
                "vra": ga_vra_effective_cnt['black']
            },
            "proportionalThreshold": {
                "raceBlind": ga_rb_rough_cnt['black'],
                "vra": ga_vra_rough_cnt['black']
            },
            "bothThreshold": {
                "raceBlind": ga_rb_both['black'],
                "vra": ga_vra_both['black']
            }
        },
        "LATINO": {
            "enactedThreshold": {
                "raceBlind": ga_rb_effective_cnt["latino"],
                "vra": ga_vra_effective_cnt["latino"]
            },
            "proportionalThreshold": {
                "raceBlind": ga_rb_rough_cnt["latino"],
                "vra": ga_vra_rough_cnt["latino"]
            },
            "bothThreshold": {
                "raceBlind": ga_rb_both["latino"],
                "vra": ga_vra_both["latino"]
            }
        },
        "OTHER": {
            "enactedThreshold": {
                "raceBlind": ga_rb_effective_cnt['other'],
                "vra": ga_vra_effective_cnt['other']
            },
            "proportionalThreshold": {
                "raceBlind": ga_rb_rough_cnt['other'],
                "vra": ga_vra_rough_cnt['other']
            },
            "bothThreshold": {
                "raceBlind": ga_rb_both['other'],
                "vra": ga_vra_both['other']
            }
        },
    },
    "state": "GA"
}

In [159]:
with open('output/Georgia/ga_impact_threshold.json', 'w') as f:
    json.dump(ga_result, f, indent=2)

# Arkansas

In [179]:
ar_rb = pd.read_json('../outputs/Arkansas/ar_raceblind_5000.jsonl', lines=True)
ar_enacted = pd.read_json('../../preprocessing/output/Arkansas/ar_enacted_effective.json')
ar_proportions = pd.read_json('../../preprocessing/output/Arkansas/ar_cvap_proportion.json')
ar_minority = ['black']

/var/folders/_7/0wq2xjx96vngcxdnd_qxp1z40000gn/T/ipykernel_30337/268679498.py:3: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  ar_proportions = pd.read_json('../../preprocessing/output/Arkansas/ar_cvap_proportion.json')
/var/folders/_7/0wq2xjx96vngcxdnd_qxp1z40000gn/T/ipykernel_30337/268679498.py:3: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  ar_proportions = pd.read_json('../../preprocessing/output/Arkansas/ar_cvap_proportion.json')
/var/folders/_7/0wq2xjx96vngcxdnd_qxp1z40000gn

In [180]:
ar_rb_effective_cnt = cnt_effective(ar_rb, ar_enacted, minorities=ar_minority, state='Arkansas')
ar_rb_rough_cnt = cnt_rough(ar_rb, enacted_rough_porportion, ar_proportions, minorities=ar_minority, total_district=4, state='Arkansas')
ar_rb_both = cnt_both(ar_rb, ar_enacted, enacted_rough_porportion, ar_proportions, minorities=ar_minority, total_district=4, state='Arkansas')
print(f'enactedThreshold:{ar_rb_effective_cnt}, proportionalThreshold:{ar_rb_rough_cnt}, bothThreshold:{ar_rb_both}')

enactedThreshold:{'black': 5000}, proportionalThreshold:{'black': 5000}, bothThreshold:{'black': 5000}


In [183]:
ar_vra = pd.read_json('../outputs/Arkansas/ar_vra_5000.jsonl', lines=True)
ar_vra_effective_cnt = cnt_effective(ar_vra, ar_enacted, minorities=ar_minority,state='Arkansas')
ar_vra_rough_cnt = cnt_rough(ar_vra, enacted_rough_porportion, ar_proportions, minorities=ar_minority, total_district=4, state='Arkansas')
ar_vra_both = cnt_both(ar_vra, ar_enacted, enacted_rough_porportion, ar_proportions, minorities=ar_minority, total_district=4, state='Arkansas')
print(f'enactedThreshold:{ar_vra_effective_cnt}, proportionalThreshold:{ar_vra_rough_cnt}, bothThreshold:{ar_vra_both}')

enactedThreshold:{'black': 5000}, proportionalThreshold:{'black': 5000}, bothThreshold:{'black': 5000}


In [184]:
ar_result = {
    "groups": {
        "BLACK": {
            "enactedThreshold": {
                "raceBlind": ar_rb_effective_cnt['black'],
                "vra": ar_vra_effective_cnt['black']
            },
            "proportionalThreshold": {
                "raceBlind": ar_rb_rough_cnt['black'],
                "vra": ar_vra_rough_cnt['black']
            },
            "bothThreshold": {
                "raceBlind": ar_rb_both['black'],
                "vra": ar_vra_both['black']
            }
        },
    },
    "state": "AR"
}

In [185]:
with open('output/Arkansas/ar_impact_threshold.json', 'w') as f:
    json.dump(ar_result, f, indent=2)